In [ ]:
lvl           = None
uods          = None
file_path_R   = None
file_path_Lvl = None
rho_delta     = None

In [ ]:
%run ./5___UOD_vals_and_lvls___Functions.ipynb

In [ ]:
uods = np.array(eval(uods))

---
---
---

In [ ]:
#uods = sorted({v for interval in uods for v in interval[:2]})
uds = sorted({uod_i[0] for uod_i in uods})
ods = sorted({uod_i[1] for uod_i in uods})

---

### Get the origins and make CDFs

For the od threshold, it makes sense to use the CDF of the origins distribution.

In [ ]:
with open(file_path_Lvl+'dict_origins_isolated.pk', 'rb') as f: origins_isolated = pkl.load(f)
with open(file_path_Lvl+'dict_origins_pairs.pk',    'rb') as f: origins_pairs    = pkl.load(f)

In [ ]:
no_origins_isolated = []; no_origins_pairs = []
for i0 in range(lvl):
    no_origins_isolated.append(len(origins_isolated[i0]))
    no_origins_pairs.append(   len(origins_pairs[   i0]))

no_origins_isolated = np.array(no_origins_isolated)
no_origins_pairs    = np.array(no_origins_pairs)
no_origins          = no_origins_isolated + no_origins_pairs

In [ ]:
cdf_origins = np.cumsum(no_origins) / np.sum(no_origins)

---

However, for the ud threshold, it makes perfect sense to use the CDF of the grid distribution.

Since we don't actually need the grid_fft density value for anything other than the analysis codes, we could skip this here... but let's not.

And since the ud_lvl is just uds[i]*lvl, since the levels here are already on the grid CDF scale, we could not even save them... but let's not.

And finally, since, like we just said, we already level based on the grid CDF scale, we wouldn't even need to calculate that CDF now... but what if someone uses a different method? Even we tried in one run to use the log scale for leveling... so let's keep this neat for the future users and just calculate that grid CDF again.

In [ ]:
with open(file_path_R+'grid_fft_delta.pk', 'rb') as f: grid_fft = pkl.load(f)

In [ ]:
grid_lvld = layer_fct_cdf(grid_fft, lvl)

---

### Get the smoothened and the cdf leveled grids

In [ ]:
for od in ods:

    # the first level (as the cdf_origins was computed on levels) that is >= uod
    # we use the first >= such that before it there is a complete od%
    od_lvl = int(np.argmax(cdf_origins >= od))
    coords = np.argwhere(grid_lvld == od_lvl)
    vals   = grid_fft[coords[:,0], coords[:,1], coords[:,2]]
    od_val = np.min(vals)

    with open(file_path_Lvl+"od_lvl___"+str(od)+".pk", 'wb') as f: pkl.dump(od_lvl, f)
    with open(file_path_Lvl+"od_val___"+str(od)+".pk", 'wb') as f: pkl.dump(od_val, f)

In [ ]:
for ud in uds:

    # the first level (as the CDF was computed on levels) that is >= ud
    # we use the first >= such that before it there is a complete ud%
    ud_lvl = int(ud * lvl)
    coords  = np.argwhere(grid_lvld == ud_lvl)
    vals    = grid_fft[coords[:,0], coords[:,1], coords[:,2]]
    ud_val = np.min(vals)

    with open(file_path_Lvl+"ud_lvl___"+str(ud)+".pk", 'wb') as f: pkl.dump(ud_lvl, f)
    with open(file_path_Lvl+"ud_val___"+str(ud)+".pk", 'wb') as f: pkl.dump(ud_val, f)

---
---
---